In [1]:
import pandas as pd

df = pd.read_csv("../dataset/disaster_tweets_processed.csv")

In [2]:
!pip install gensim


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)


y_train = df.loc[train_idx, "target"]
y_test = df.loc[test_idx, "target"]

In [4]:
train_sentences_clean = [
    text.split()
    for text in df.loc[train_idx, "clean_text"]
]

In [5]:
from gensim.models import Word2Vec

w2v_clean = Word2Vec(
    train_sentences_clean,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=20
)

In [6]:
train_sentences_lemma = [
    text.split()
    for text in df.loc[train_idx, "lemma_text"]
]

w2v_lemma = Word2Vec(
    train_sentences_lemma,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=20
)

In [7]:
import numpy as np

def avg_vector(text, model):

    words = text.split()

    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [8]:
X_clean = np.array([
    avg_vector(text, w2v_clean)
    for text in df["clean_text"]
])

In [9]:
X_lemma = np.array([
    avg_vector(text, w2v_lemma)
    for text in df["lemma_text"]
])

In [10]:
X_train_clean = X_clean[train_idx]
X_test_clean = X_clean[test_idx]

X_train_lemma = X_lemma[train_idx]
X_test_lemma = X_lemma[test_idx]

In [11]:
from sklearn.model_selection import cross_val_score

In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)


scores = cross_val_score(
    lr,
    X_train_lemma,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.7188113653101521


In [13]:
from sklearn.svm import LinearSVC

svm = LinearSVC()

scores = cross_val_score(
    svm,
    X_train_clean,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.732955564576299


In [14]:
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()

scores = cross_val_score(
    nb,
    X_train_lemma,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.6426678298239721


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(min_df=2)

tfidf.fit(
    df.loc[train_idx, "lemma_text"]
)

tfidf_vocab = tfidf.vocabulary_
idf_scores = dict(zip(
    tfidf.get_feature_names_out(),
    tfidf.idf_
))

In [16]:
import numpy as np

def weighted_avg_vector(text, w2v_model):

    words = text.split()

    weighted_vectors = []
    weights = []

    for word in words:

        if word in w2v_model.wv and word in idf_scores:

            weight = idf_scores[word]

            weighted_vectors.append(
                w2v_model.wv[word] * weight
            )

            weights.append(weight)

    if len(weighted_vectors) == 0:
        return np.zeros(w2v_model.vector_size)

    return np.sum(weighted_vectors, axis=0) / np.sum(weights)

In [17]:
X_weighted = np.array([
    weighted_avg_vector(text, w2v_lemma)
    for text in df["lemma_text"]
])

In [18]:
print(X_weighted.shape)

(7613, 300)


In [19]:
X_train = X_weighted[train_idx]
X_test = X_weighted[test_idx]

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr = LogisticRegression(max_iter=1000)

scores = cross_val_score(
    lr,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.7216551863239232


In [21]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [22]:
tagged_data = [
    TaggedDocument(
        words=text.split(),
        tags=[str(i)]
    )
    for i, text in enumerate(
        df.loc[train_idx, "clean_text"]
    )
]

In [23]:
doc2vec_model = Doc2Vec(
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    epochs=50,
    dm=1
)

doc2vec_model.build_vocab(tagged_data)

doc2vec_model.train(
    tagged_data,
    total_examples=doc2vec_model.corpus_count,
    epochs=doc2vec_model.epochs
)

In [24]:
import numpy as np

X_doc2vec = np.array([
    doc2vec_model.infer_vector(text.split())
    for text in df["clean_text"]
])

In [25]:
print(X_doc2vec.shape)

(7613, 300)


In [26]:


X_train = X_doc2vec[train_idx]
X_test = X_doc2vec[test_idx]

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr = LogisticRegression(max_iter=1000)

scores = cross_val_score(
    lr,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print(scores.mean())

0.6023327688431278
